# Stage 4: tracked experiments + TensorBoard
Q19 map: **log_params** = settings, **log_metrics** = final numbers, **log_time_series_metrics** = loss per epoch
(this is what TensorBoard plots), **assign_input_artifacts** = the data the run used (an artifact, never a metric).
Two runs on the same data: XGBoost (no curve, just final metrics) and Keras (loss curve).

In [ ]:
%pip -q install xgboost db-dtypes scikit-learn
# after this cell FIRST runs, restart the kernel once (Kernel -> Restart) so imports see the new packages
import tensorflow as tf, xgboost as xgb, numpy as np, pandas as pd
from scipy.stats import spearmanr
from google.cloud import aiplatform
print(tf.__version__, xgb.__version__, aiplatform.__version__)

## 1. Create the TensorBoard instance (one-time)
**Cost:** Vertex TensorBoard bills only for stored logs (about $10 per GiB per month, as of the 2023 pricing change). This lab logs kilobytes, so it is negligible. The Workbench instance is the real cost: stop it when done.

In [ ]:
PROJECT, REGION = "fantasy-football-ml-503718", "us-central1"
aiplatform.init(project=PROJECT, location=REGION)
existing = aiplatform.Tensorboard.list(filter='display_name="nfl-lab-tb"')
tb = existing[0] if existing else aiplatform.Tensorboard.create(display_name="nfl-lab-tb")
print(tb.resource_name)

## 2. Start the experiment, linked to that TensorBoard

In [ ]:
aiplatform.init(project=PROJECT, location=REGION, experiment="nfl-lab",
                experiment_tensorboard=tb)

## 3. Read the data from BigQuery and make TRAIN / VALID matrices
Split is by season (the `split` column from `v_split`). Missing values are filled with the TRAIN median only, so nothing leaks from 2024.

In [ ]:
from google.cloud import bigquery
df = bigquery.Client(project=PROJECT).query("SELECT * FROM nfl_lab.v_split").to_dataframe(create_bqstorage_client=False)
DROP = ["player_id","player_display_name","recent_team","opponent_team","season","week","dk_points","split"]
X_all = pd.get_dummies(df.drop(columns=DROP), columns=["position"], dtype=float).astype(float)
tr, va = df.split == "TRAIN", df.split == "VALID"
med = X_all[tr].median()
Xtr, Xva = X_all[tr].fillna(med), X_all[va].fillna(med)
ytr, yva = df.loc[tr,"dk_points"], df.loc[va,"dk_points"]
print(Xtr.shape, Xva.shape)

def weekly_spearman(pred):
    d = pd.DataFrame({"s":df.loc[va,"season"],"w":df.loc[va,"week"],"y":yva.values,"p":pred})
    return d.groupby(["s","w"]).apply(lambda g: spearmanr(g.y,g.p)[0]).mean()

## 4. Record the data as an ARTIFACT (this is the `assign_input_artifacts` part of Q19)

In [ ]:
ds_art = aiplatform.Artifact.create(schema_title="system.Dataset", display_name="nfl_lab.v_split",
                                    uri=f"bq://{PROJECT}.nfl_lab.v_split")
print(ds_art.name)

## 5. Run 1: XGBoost - params + final metrics, no curve

In [ ]:
with aiplatform.start_run("xgb-baseline-01") as run:
    with aiplatform.start_execution(schema_title="system.ContainerExecution", display_name="train-xgb") as ex:
        ex.assign_input_artifacts([ds_art])
        params = dict(n_estimators=600, max_depth=5, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, min_child_weight=20)
        aiplatform.log_params(params)
        m = xgb.XGBRegressor(**params, n_jobs=-1).fit(Xtr, ytr)
        p = m.predict(Xva)
        aiplatform.log_metrics({"val_mae": float(np.mean(np.abs(p-yva))), "val_weekly_spearman": float(weekly_spearman(p))})
print("done")

## 6. Run 2: Keras - loss logged every epoch as a TIME SERIES -> TensorBoard

In [ ]:
mu, sd = Xtr.mean(), Xtr.std().replace(0,1)
Atr, Ava = ((Xtr-mu)/sd).values, ((Xva-mu)/sd).values

class LogToVertex(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        aiplatform.log_time_series_metrics({"loss": logs["loss"], "val_loss": logs["val_loss"]}, step=epoch)

with aiplatform.start_run("keras-mlp-01") as run:
    with aiplatform.start_execution(schema_title="system.ContainerExecution", display_name="train-keras") as ex:
        ex.assign_input_artifacts([ds_art])
        aiplatform.log_params({"model":"mlp","layers":"64-32","lr":1e-3,"epochs":40,"batch":256})
        net = tf.keras.Sequential([tf.keras.layers.Input((Atr.shape[1],)),
                                   tf.keras.layers.Dense(64,activation="relu"),
                                   tf.keras.layers.Dense(32,activation="relu"),
                                   tf.keras.layers.Dense(1)])
        net.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
        net.fit(Atr, ytr.values, validation_data=(Ava, yva.values), epochs=40, batch_size=256,
                callbacks=[LogToVertex()], verbose=2)
        p = net.predict(Ava, verbose=0).ravel()
        aiplatform.log_metrics({"val_mae": float(np.mean(np.abs(p-yva))), "val_weekly_spearman": float(weekly_spearman(p))})
print("done")

## 7. Compare runs in one table
Then open the console: **Agent Platform -> Experiments -> nfl-lab -> keras-mlp-01 -> Open TensorBoard** to see the loss curve.

In [ ]:
aiplatform.get_experiment_df("nfl-lab")

## 8. Clean up when finished
Deletes the TensorBoard instance and its stored logs. Optional tidiness, since the cost is negligible.

In [ ]:
# tb.delete()